In [87]:
import meshio as mio
import numpy as np
import igl
import copy

# test mesh

In [88]:
tet_vertices = np.array([
    [0,0,0],
    [1,0,0],
    [0,1,0],
    [-1,0,0],
    [0,0,1]
])

tets = np.array([
    [0,1,2,4],
    [0,2,3,4]
])

init_tetmesh = mio.Mesh(tet_vertices, [('tetra', tets)])
init_tetmesh.write("init_tetmesh.msh", file_format='gmsh')

surface_vertices = np.array([
    [0,0,0],
    [1,0,0],
    [0,1,0],
    [-1,0,0]
])

tris = np.array([
    [0,1,2],
    [0,2,3]
])

surface_v_to_tet_v_map = {}
for i in range(4):
    surface_v_to_tet_v_map[i] = i

surface_adj_tets = {}
surface_adj_tets[0] = [0]
surface_adj_tets[1] = [1]

tet_surface = {}

for i in range(2):
    for tet_id in surface_adj_tet[i]:
        tet_surface[tet_id] = i


Warning: Binary Gmsh needs c_double points (got int64). Converting.

In [89]:
para_vertices = np.array([
    [0,0,0],
    [1,0,0],
    [0,1,0],
    [-1,0,0],
    [0,0.5,0],
    [0.5,0,0]
])

para_tris = np.array([
    [0,5,4],
    [4,5,2],
    [5,1,2],
    [0,4,3],
    [4,2,3]
])

para_in_to_out_face_map = {}
para_in_to_out_face_map[0] = [0,1,2]
para_in_to_out_face_map[1] = [3,4]

# split tet surface

In [90]:
old_surface_v_cnt = surface_vertices.shape[0]
new_surface_vs = para_vertices[old_surface_v_cnt:, ]

# add new vertices to tets and update map
for i in range(new_surface_vs.shape[0]):
    new_idx = tet_vertices.shape[0]
    tet_vertices = np.append(tet_vertices, [new_surface_vs[i]], axis=0)
    surface_v_to_tet_v_map[i + old_surface_v_cnt] = new_idx


In [91]:
new_tets = []
keep_flag = [True] * tets.shape[0]
new_surface_adj_tets = {}

for i in range(tris.shape[0]):
    if len(para_in_to_out_face_map[i]) == 1:
        # not splitted
        new_surface_adj_tets[para_in_to_out_face_map[i][0]] = surface_adj_tets[i]
        continue
    else:
        splitted_faces = para_in_to_out_face_map[i]
        splitted_faces_in_tet_vid = [[surface_v_to_tet_v_map[vid] for vid in para_tris[f]] for f in splitted_faces]

        # print(splitted_faces_in_tet_vid)

        for tet_id in surface_adj_tets[i]:
            # mark as tet to drop
            keep_flag[i] = False

            tet = tets[tet_id]
            mapped_tri = [surface_v_to_tet_v_map[k] for k in tris[i]]
            # find the apex 
            apex = -1
            for vid in tet:
                if vid not in mapped_tri:
                    apex = vid
            assert apex != -1

            for k, spf in enumerate(splitted_faces_in_tet_vid):

                new_surface_adj_tets[splitted_faces[k]] = len(new_tets)
                new_tets.append([spf[0], spf[1], spf[2], apex])


final_tets = []

old_tid_to_new_tid_map = {}

final_tet_cnt = 0
for i in range(len(keep_flag)):
    if keep_flag[i]:
        final_tets.append(tets[i])
        old_tid_to_new_tid_map[i] = final_tet_cnt

    final_tet_cnt += 1

for i in range(len(new_tets)):
    old_tid_to_new_tid_map[i + tets.shape[0]] = i + final_tet_cnt

for i in range(len(new_tets)):
    final_tets.append(np.array(new_tets[i]))

final_tets = np.array(final_tets)

# fix orientation
final_tets_oriented = []

for tet in final_tets:
    if igl.predicate.orient3d(tet_vertices[tet[0]], tet_vertices[tet[1]], tet_vertices[tet[2]], tet_vertices[tet[3]]) <= 0:
        final_tets_oriented.append(np.array([tet[1], tet[0], tet[2], tet[3]]))
    else:
        final_tets_oriented.append(tet)

final_tets_oriented = np.array(final_tets_oriented)
        



AttributeError: module 'igl' has no attribute 'predicate'

In [ ]:
new_tetmesh = mio.Mesh(tet_vertices, [("tetra", final_tets)])

In [ ]:
new_tetmesh.write("para_splitted_tets.msh", file_format='gmsh')


In [ ]:
final_tets

array([[0, 6, 5, 4],
       [5, 6, 2, 4],
       [6, 1, 2, 4],
       [0, 5, 3, 4],
       [5, 2, 3, 4]])

In [ ]:
tet_vertices

array([[ 0. ,  0. ,  0. ],
       [ 1. ,  0. ,  0. ],
       [ 0. ,  1. ,  0. ],
       [-1. ,  0. ,  0. ],
       [ 0. ,  0. ,  1. ],
       [ 0. ,  0.5,  0. ],
       [ 0.5,  0. ,  0. ]])